[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlatcl/multilogue/blob/main/notebooks/09_formats_complexity.ipynb)

# 09 · Formats-complexity: public Access + full Address stack

Notebooks 00–08 teach pieces. This one asks whether multilogue can show the **full Address complexity** of an [`ai-dialogue-formats`](https://github.com/mlatcl/ai-dialogue-formats)-style study: two *formats* of convening, multi-kind extraction, multi-run lenses, document-weighted hard and soft format contrasts, σ² neighbour rules, document EMD / PERMANOVA, and a multiverse on a **format-gap** ℓ.

**Access (public data):** CIMT Bonn *cycling dialogue* vs *survey* ([cimt-argument-mining-dataset](https://github.com/juliaromberg/cimt-argument-mining-dataset)) stand in for workshop vs survey / chatbot vs workshop. Units are tagged `format=dialogue|survey`.

**Address (offline):** German CIMT text does not match the English `FakeClient` phrase bank, so the runnable Address path uses **format-tagged synthetic English units** with the same group mechanics. With a real LLM client you can swap in the CIMT units from Access.

Requires optional ``pot`` for soft + distance (`pip install 'multilogue[distance]'`).

In [ ]:
#@title Setup (run first)
# Free path: synthetic units + FakeClient (no API key).
# Optional: load("polis", ...) downloads public open data into a local cache.

import sys, subprocess, pathlib

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "matplotlib", "pandas", "numpy", "scikit-learn", "pyyaml",
        "git+https://github.com/mlatcl/multilogue.git",
    ])
else:
    try:
        import multilogue  # noqa: F401
    except ImportError:
        _here = pathlib.Path.cwd().resolve()
        for _cand in (_here, _here.parent, *_here.parents):
            if (_cand / "pyproject.toml").is_file() and (_cand / "multilogue").is_dir():
                if str(_cand) not in sys.path:
                    sys.path.insert(0, str(_cand))
                break

from IPython.display import display, HTML, Markdown
from multilogue.notebook import setup

FIG = setup(figures_dir="outputs/notebook_figures")
print("Figures →", FIG)


## Access — CIMT dialogue vs survey

Assess without the analysis question: coverage by format and document counts. Download goes to `MULTILOGUE_DATAPATH` or `~/.cache/multilogue` (set `MULTILOGUE_ACCESS_YES=1`). If the network is unavailable, we fall back to a short synthetic Access table so the notebook still runs.

In [ ]:
import os
import pandas as pd
from multilogue.experiment import load_cimt_format_pair, format_tagged_units
from multilogue.assess_base import docs_per_group, unit_size_table

os.environ.setdefault("MULTILOGUE_ACCESS_YES", "1")
cimt_units, cimt_meta = load_cimt_format_pair(max_units_per=60)
if cimt_units is None:
    print("CIMT unavailable:", cimt_meta.get("error", "unknown"))
    access_units = format_tagged_units(n_docs_per_group=3, units_per_doc=2)
    access_label = "synthetic format stand-in (Access fallback)"
else:
    access_units = cimt_units
    access_label = "CIMT Cycling-Dialogue-Bonn vs Survey-Bonn"
    display(cimt_meta)

print(access_label, "—", len(access_units), "units")
cov = docs_per_group(access_units, "format")
sizes = unit_size_table(access_units)
display(cov)
display(sizes.describe())
if cimt_units:
    print("Sample dialogue:", next(u["text"][:120] for u in cimt_units if u["format"] == "dialogue"))
    print("Sample survey:", next(u["text"][:120] for u in cimt_units if u["format"] == "survey"))
    codes = (
        pd.DataFrame([{"format": u["format"], "code": (u.get("meta") or {}).get("code")}
                      for u in cimt_units])
        .value_counts()
        .rename("n")
        .reset_index()
    )
    print("CIMT gold argument codes (already on units — not LLM extract):")
    display(codes)

## Address — formats-complexity demo (FakeClient, always runs)

`run_formats_complexity_demo` wires the shared method end-to-end with `format` as the group column: multi-kind extract, frozen clustering, multi-run lenses, hard `group_compare`, soft PPCA + `soft_group_compare` + σ² rules, document distance, and multiverse on a format-gap ℓ.

Checkpoints land under `outputs/…/checkpoints/` (gitignored). Re-running skips completed fingerprints until prompt/model/version changes.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pot"])
from pathlib import Path
from multilogue.experiment import run_formats_complexity_demo
from multilogue import viz
import pandas as pd

out = Path("outputs/notebook_formats_complexity")
summary = run_formats_complexity_demo(
    out, n_docs_per_group=4, units_per_doc=3, k=5, lens_runs=3, quiet=True
)
display(summary)
tables = out / "tables"
for name in [
    "yield.csv",
    "group_compare_format_lenses.csv",
    "soft_group_compare_format.csv",
    "sigma2_rules.csv",
    "doc_distance_pairwise.csv",
    "variance_shares_format_gap.csv",
]:
    p = tables / name
    if p.is_file():
        print("—", name)
        display(pd.read_csv(p).head(12))

shares = pd.read_csv(tables / "variance_shares_format_gap.csv")
viz.variance_share_bars(shares, path=FIG / "09_format_gap_variance.png")
print("Evidence pack:", out / "evidence_pack.html")
print("Soft:", summary.get("soft"))
print("Distance:", summary.get("distance"))

## Findings — clusters, concerns, format contrasts

Readable summary from the Address tables (no extra LLM calls). Point `TABLES` at the run you care about — live CIMT by default when present.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display
from multilogue.report import (
    cluster_catalogue, findings_markdown, top_phrases_by_group, write_findings,
)

_root_cand = Path("outputs/formats_cimt_llm/tables")
_synth = Path("outputs/notebook_formats_complexity/tables")
TABLES = _root_cand if (_root_cand / "phrases_clustered.csv").is_file() else _synth
print("Reading", TABLES.resolve())

ph = pd.read_csv(TABLES / "phrases_clustered.csv")
cmp = (
    pd.read_csv(TABLES / "group_compare_format_lenses.csv")
    if (TABLES / "group_compare_format_lenses.csv").is_file()
    else None
)
multi = (
    pd.read_csv(TABLES / "multiverse_format_gap.csv")
    if (TABLES / "multiverse_format_gap.csv").is_file() else None
)
shares = (
    pd.read_csv(TABLES / "variance_shares_format_gap.csv")
    if (TABLES / "variance_shares_format_gap.csv").is_file() else None
)
cat = cluster_catalogue(ph)
display(Markdown("### Cluster catalogue"))
display(cat)
display(Markdown("### Top concerns by format"))
for g, tab in top_phrases_by_group(ph, kind="concern", n=8).items():
    display(Markdown(f"**{g}**"))
    display(tab)
if shares is not None:
    display(Markdown("### Multiverse variance shares (which decisions move ℓ)"))
    display(shares)
if multi is not None:
    display(Markdown("### Multiverse grid (ℓ = largest |format gap| under each config)"))
    display(multi.head(20))
md = findings_markdown(
    ph, compare=cmp, multiverse=multi, variance_shares=shares,
    title="CIMT dialogue vs survey — findings",
)
write_findings(
    ph, TABLES / "findings.md", compare=cmp,
    multiverse=multi, variance_shares=shares,
    title="CIMT dialogue vs survey — findings",
)
display(Markdown(md))
print("Wrote", TABLES / "findings.md")

## Diagnostics — σ² curve, mixture weights, document MDS

These plots read the same `TABLES` directory as Findings.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from multilogue import viz
from IPython.display import Markdown, display

if "TABLES" not in dir():
    _root_cand = Path("outputs/formats_cimt_llm/tables")
    _synth = Path("outputs/notebook_formats_complexity/tables")
    TABLES = _root_cand if (_root_cand / "phrases_clustered.csv").is_file() else _synth

curve_p = TABLES / "sigma2_stability_curve.csv"
W_p = TABLES / "lens_mixture_weights.csv"
sum_p = TABLES / "summary.json"
if curve_p.is_file():
    curve = pd.read_csv(curve_p)
    star = None
    if sum_p.is_file():
        import json
        star = json.loads(sum_p.read_text()).get("soft", {}).get("sigma2_best")
    viz.sigma2_stability_curve(curve, sigma2_best=star, path=FIG / "09_sigma2_curve.png")
    display(curve.head(6))
else:
    print("No sigma2_stability_curve.csv yet")

if W_p.is_file():
    Wdf = pd.read_csv(W_p, index_col=0)
    viz.mixture_weight_heatmap(
        Wdf.to_numpy(dtype=float), list(Wdf.columns),
        path=FIG / "09_mixture_weights.png",
    )
    display(Wdf.round(3))
    if Wdf.shape[1] < 2:
        display(Markdown(
            "**Degenerate soft layer:** only one lens column. "
            "Usually label/lens LLM stages failed (check `logs/*_errors.json`). "
            "σ² sweep then cannot enter the meaningful-soft band."
        ))
else:
    print("No lens_mixture_weights.csv yet")

In [ ]:
# Document MDS from phrases_clustered + recompute EMD (needs pot + centroids npy if saved).
# Pairwise PERMANOVA table from the run:
pair_p = TABLES / "doc_distance_pairwise.csv"
if pair_p.is_file():
    display(pd.read_csv(pair_p))
else:
    print("No doc_distance_pairwise.csv yet")

ph_p = TABLES / "phrases_clustered.csv"
if ph_p.is_file():
    ph = pd.read_csv(ph_p)
    viz.cluster_size_bars(ph, path=FIG / "09_cluster_sizes.png")
    if "lens" in ph.columns:
        print("Lens counts:", ph["lens"].value_counts().to_dict())
    cmp_p = TABLES / "group_compare_format_lenses.csv"
    if cmp_p.is_file():
        cmp = pd.read_csv(cmp_p)
        display(cmp)
        # dumbbell needs group column names — soft/hard tables vary
        gcols = [c for c in cmp.columns if c.endswith("_pct")]
        if len(gcols) >= 2 and "lens" in cmp.columns:
            ga, gb = gcols[0].removesuffix("_pct"), gcols[1].removesuffix("_pct")
            viz.group_compare_dumbbell(
                cmp, label_col="lens", group_a=ga, group_b=gb,
                path=FIG / "09_format_dumbbell.png",
            )

## Optional — live LLM on CIMT

### Where to put `.env`

Put it at the **repository root** (`multilogue/.env`), not under `notebooks/`:

```bash
cp .env.example .env   # from repo root
# edit: OPENAI_API_KEY=…
```

`load_dotenv()` looks at `MULTILOGUE_ROOT/.env` then **cwd** `.env`. If you launch Jupyter from `notebooks/`, either keep `.env` at repo root and set `MULTILOGUE_ROOT`, or rely on the cell below (it walks up to the repo root). Never commit `.env`.

Checkpoints → `outputs/formats_cimt_llm/checkpoints/` (gitignored). Do not commit JSONL with unit text. This section no-ops without a key.

### Your analysis choices (edit the next code cell)

These are the study-local knobs an `ai-dialogue-formats`-style package would own. Fill them in before the live run:

1. **Extract kinds** — what the model pulls from each unit  
   - `concern` / `benefit` — formats analogy (cycling-adapted prompts)  
   - `claim` / `premise` — closer to CIMT gold codes already on units  
   - or name your own tuple
2. **Lens hint** — optional framing names for the lens prompt (e.g. safety, infrastructure, equity); leave empty for free proposal
3. **k / lens_runs / max units** — cost vs coverage for the first paid pass
4. **Headline ℓ** — today the demo uses a toy Lens-A share gap; swap later for experience-style or claim-rate gaps once grounds coding exists

CIMT gold `code` (premise / claim / non-arg) is already on unit `meta` from Access — useful for validation, not a substitute for choosing extract kinds.

In [ ]:
# --- edit these before enabling the live run ---
KINDS = ("concern", "benefit")  # or ("claim", "premise") or your own
LENS_HINT = "Prefer framings among: safety, infrastructure, access/connectivity, equity, other."
MAX_UNITS_PER_FORMAT = 40  # raise after a cheap pilot
DROP_NON_ARG = True  # CIMT gold code == non-arg (often titles) → skip in Address
K_CLUSTERS = 6
N_LENS_RUNS = 3
EXTRACT_MODEL = "gpt-4.1-mini"
EMBED_MODEL = "text-embedding-3-small"
EXTRACT_SYSTEM = (
    "You extract short phrases from German civic comments about cycling. "
    "For concern: only problems, risks, barriers, or complaints — not proposals or wishes. "
    "For benefit: only advantages or positive outcomes already claimed — not bare demands. "
    "Keep the phrase in the source language. "
    "Reply with bullet lines '- phrase || supporting quote' or NO_<KIND>."
)
print("Configured:", {
    "KINDS": KINDS, "LENS_HINT": LENS_HINT or "(free proposal)",
    "MAX_UNITS_PER_FORMAT": MAX_UNITS_PER_FORMAT, "DROP_NON_ARG": DROP_NON_ARG,
    "K_CLUSTERS": K_CLUSTERS, "N_LENS_RUNS": N_LENS_RUNS,
    "EXTRACT_MODEL": EXTRACT_MODEL, "EMBED_MODEL": EMBED_MODEL,
})

In [ ]:
import os
from pathlib import Path
from multilogue.config import load_dotenv

# Prefer repo-root .env even if the kernel cwd is notebooks/
_here = Path.cwd().resolve()
_root = next(
    (p for p in [_here, *_here.parents] if (p / "pyproject.toml").is_file()
     and (p / "multilogue").is_dir()),
    _here,
)
loaded = load_dotenv(_root / ".env") or load_dotenv()
print("dotenv:", loaded or "(none found — set OPENAI_API_KEY in the environment)")
print(
    "Cache behaviour: StageRunner skips keys already in checkpoints/*.jsonl "
    "with the same fingerprint; embeddings resume by phrase hash. "
    "Failed items are not written as success — they retry on the next run."
)

if not os.environ.get("OPENAI_API_KEY"):
    print("No OPENAI_API_KEY — skip live CIMT Address.")
elif not cimt_units:
    print("No CIMT units loaded — skip live path.")
else:
    from multilogue.experiment import load_cimt_format_pair, run_formats_complexity_demo
    from multilogue.openai_client import OpenAIClient

    live_units, live_meta = load_cimt_format_pair(max_units_per=MAX_UNITS_PER_FORMAT)
    if live_units is None:
        print("CIMT reload failed:", live_meta.get("error"))
    else:
        if DROP_NON_ARG:
            before = len(live_units)
            live_units = [
                u for u in live_units
                if (u.get("meta") or {}).get("code") != "non-arg"
            ]
            print(f"DROP_NON_ARG: {before} → {len(live_units)} units")

        def user_for_unit(u):
            return u.get("extract_text") or u["text"]

        if LENS_HINT.strip():
            print("Lens hint (for your notes / later edited YAML):", LENS_HINT)

        llm_out = Path("outputs/formats_cimt_llm")
        if not llm_out.is_absolute():
            llm_out = _root / llm_out
        client = OpenAIClient(log_path=llm_out / "logs" / "openai.jsonl")
        live = run_formats_complexity_demo(
            llm_out,
            units=live_units,
            client=client,
            kinds=KINDS,
            extract_model=EXTRACT_MODEL,
            embed_model=EMBED_MODEL,
            label_model=EXTRACT_MODEL,
            k=K_CLUSTERS,
            lens_runs=N_LENS_RUNS,
            extract_system=EXTRACT_SYSTEM,
            extract_user_for_unit=user_for_unit,
            quiet=False,
            max_workers=4,
        )
        display(live)
        print("Checkpoints:", live["checkpoint_dir"])
        if live.get("lens_stage_missing") or live.get("n_distinct_lenses", 0) < 2:
            print(
                "WARNING: lens stage incomplete or collapsed. "
                "Inspect", llm_out / "logs" / "lenses_k*_errors.json",
                "then re-run this cell (extract/embed cache will resume)."
            )
        TABLES = llm_out / "tables"  # so diagnostic cells above can be re-run

## What stays in a consumer package

Multilogue owns the shared stages above. An `ai-dialogue-formats`-style package still keeps: PDF/docx Access, study prompts and experience coding, researcher-edited lens YAML, and the precise definition of headline ℓ (e.g. experience-gap rather than the toy Lens-A share used here).

See the [formats cookbook](https://github.com/mlatcl/multilogue/blob/main/docs/cookbooks/ai-dialogue-formats.md) and [soft / EMD concepts](https://github.com/mlatcl/multilogue/blob/main/docs/concepts/soft-ppca-emd.md).